# Encoder Comparison -- WavLM vs Wav2Vec2 vs Whisper-medium

Plug-and-play notebook:
1. Fill `TRAIN_FOLDERS` and (optionally) `TEST_FOLDERS` below.
2. Each folder must have `{foldername}GT.csv` with columns `filename`, `label`, and a speaker-id column (auto-detected from common names: `speaker_id`, `speaker`, `spk`, `spkr`, `student_id`). If none of those exist, the filename itself is treated as the speaker id (= no real isolation, a warning is printed).
3. Run all cells. For each of {WavLM-base-plus, Wav2Vec2-base, Whisper-medium-encoder}, the notebook:
   - autodetects the cached embedding CSV; only extracts for missing files
   - builds a **speaker-isolated** train/test split (no speaker appears on both sides)
   - trains XGBoost and reports precision/recall/F1 + threshold sweep
4. Final cell prints a side-by-side comparison table.

### Speaker isolation rules
- If `TEST_FOLDERS` is non-empty: every speaker that appears in any test folder is **removed** from the train set (with a warning printed) so there is zero overlap.
- If `TEST_FOLDERS` is empty: `GroupShuffleSplit` is used on all data with `TEST_RATIO` (default 0.2) and `groups = speaker_id`.

In [ ]:
# ================================================================
# CONFIGURATION -- just edit this cell
# ================================================================

TRAIN_FOLDERS = [
    r"../audios2",
    r"../audios4",
]

# Optional. Leave empty list to auto-split 0.2 of train (speaker-isolated).
TEST_FOLDERS = [
    r"../audios5",
]

TEST_RATIO   = 0.20
RANDOM_SEED  = 42

# Speaker column hints in GT CSVs (case-insensitive; first match wins).
SPEAKER_COL_CANDIDATES = [
    "speaker_id", "speaker", "spk", "spkr",
    "student_id", "candidate_id", "user_id",
]

# Encoder model names (HF hub ids). Order here = run order in the notebook.
MODELS = [
    ("wavlm",    "microsoft/wavlm-base-plus"),
    ("wav2vec2", "facebook/wav2vec2-base"),
    ("whisper",  "openai/whisper-medium"),
]

# Whisper processes audio in 30s chunks; we mean-pool across chunks.
WHISPER_CHUNK_SEC = 30

# XGBoost hyperparameters -- same as the 4-way comparison setup.
XGB_PARAMS = dict(
    n_estimators=400, max_depth=5, learning_rate=0.04,
    subsample=0.8, min_child_weight=3,
    eval_metric='logloss', random_state=RANDOM_SEED,
)

LABEL_MAP = {
    'read':1,'cheating':1,'reading':1,'scripted':1,'yes':1,'1':1,1:1,
    'spontaneous':0,'not cheating':0,'not_cheating':0,'no':0,'0':0,0:0,'genuine':0,
}

AUDIO_EXTS = {".wav", ".flac", ".mp3", ".m4a", ".ogg"}

print(f"Train folders: {TRAIN_FOLDERS}")
print(f"Test folders : {TEST_FOLDERS if TEST_FOLDERS else '(auto-split 0.2)'}")

In [ ]:
import os, sys, json, warnings, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import soundfile as sf
import librosa
from tqdm import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    roc_auc_score, confusion_matrix,
)
import xgboost as xgb
import joblib

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

NB_DIR   = Path('.').resolve()
SAVE_DIR = NB_DIR / 'checkpoints_encoder_cmp'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Scan folders + build master index (filename, label, speaker, role)

In [ ]:
def find_speaker_col(df):
    lower = {c.lower(): c for c in df.columns}
    for cand in SPEAKER_COL_CANDIDATES:
        if cand in lower:
            return lower[cand]
    return None

def normalize_label(x):
    return LABEL_MAP.get(x, LABEL_MAP.get(str(x).lower().strip(), -1))

def load_folder_index(folder_path, role):
    p = Path(folder_path).resolve()
    if not p.exists():
        print(f"  [SKIP] {folder_path} does not exist")
        return pd.DataFrame()
    name = p.name
    gt_path = p.parent / f'{name}GT.csv'
    if not gt_path.exists():
        gt_path = p / f'{name}GT.csv'
    if not gt_path.exists():
        print(f"  [SKIP] {name}: no {name}GT.csv found next to the folder")
        return pd.DataFrame()

    gt = pd.read_csv(gt_path)
    fn_col = next((c for c in gt.columns if c.lower() in ('filename','file','name')), gt.columns[0])
    lb_col = next((c for c in gt.columns if c.lower() in ('label','class','cheating','gt','label_int','ground_truth')), None)
    if lb_col is None:
        print(f"  [SKIP] {name}: no label column in GT CSV")
        return pd.DataFrame()

    sp_col = find_speaker_col(gt)
    if sp_col is None:
        print(f"  [WARN] {name}: no speaker column found (tried {SPEAKER_COL_CANDIDATES}). Falling back to filename-as-speaker -- SPEAKER ISOLATION DEGRADES TO FILE ISOLATION in this folder.")

    gt = gt.rename(columns={fn_col:'filename', lb_col:'label_raw'})
    gt['label'] = gt['label_raw'].apply(normalize_label)
    gt = gt[gt['label'].isin([0,1])].copy()
    gt['speaker_id'] = gt[sp_col].astype(str) if sp_col else gt['filename'].astype(str)
    gt['folder']     = name
    gt['role']       = role
    gt['filepath']   = gt['filename'].apply(lambda fn: str(p / fn))

    existing = {f.name for f in p.iterdir() if f.is_file() and f.suffix.lower() in AUDIO_EXTS}
    mask = gt['filename'].isin(existing)
    if (~mask).any():
        print(f"  [WARN] {name}: {(~mask).sum()} files in GT not found on disk, dropping")
        gt = gt[mask].copy()

    print(f"  {name:<20} role={role:<5} n={len(gt):>4}  speakers={gt['speaker_id'].nunique():>4}  "
          f"cheat={(gt['label']==1).sum():>4}  honest={(gt['label']==0).sum():>4}")
    return gt[['filepath','filename','label','speaker_id','folder','role']]

print('Loading train folders:')
train_parts = [load_folder_index(f, 'train') for f in TRAIN_FOLDERS]
train_parts = [t for t in train_parts if len(t)]
print('\nLoading test folders:')
test_parts  = [load_folder_index(f, 'test')  for f in TEST_FOLDERS]
test_parts  = [t for t in test_parts if len(t)]

assert train_parts, 'No valid train data. Check paths and GT files.'

all_df = pd.concat(train_parts + test_parts, ignore_index=True)
print(f'\nTotal files indexed: {len(all_df)}  unique speakers: {all_df["speaker_id"].nunique()}')

## 2. Speaker-isolated train/test split

In [ ]:
if test_parts:
    test_df  = pd.concat(test_parts, ignore_index=True)
    train_df = pd.concat(train_parts, ignore_index=True)
    leak_speakers = set(test_df['speaker_id']) & set(train_df['speaker_id'])
    if leak_speakers:
        print(f'[isolation] {len(leak_speakers)} speakers appear in both sides; removing them from TRAIN.')
        before = len(train_df)
        train_df = train_df[~train_df['speaker_id'].isin(leak_speakers)].copy()
        print(f'[isolation] train rows: {before} -> {len(train_df)}')
    else:
        print('[isolation] No speaker overlap between train/test folders. Clean split.')
else:
    train_df = pd.concat(train_parts, ignore_index=True)
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_RATIO, random_state=RANDOM_SEED)
    tr_idx, te_idx = next(gss.split(train_df, groups=train_df['speaker_id']))
    test_df  = train_df.iloc[te_idx].copy()
    train_df = train_df.iloc[tr_idx].copy()
    print(f'[auto-split] test_ratio={TEST_RATIO}  (group-aware)')

overlap = set(train_df['speaker_id']) & set(test_df['speaker_id'])
assert not overlap, f'speaker leak: {overlap}'
print(f'\nTrain: {len(train_df):>4}  ({train_df["speaker_id"].nunique()} speakers)  '
      f'cheat={(train_df["label"]==1).sum()}  honest={(train_df["label"]==0).sum()}')
print(f'Test : {len(test_df):>4}  ({test_df["speaker_id"].nunique()} speakers)  '
      f'cheat={(test_df["label"]==1).sum()}  honest={(test_df["label"]==0).sum()}')

spw = (train_df['label']==0).sum() / max((train_df['label']==1).sum(), 1)
print(f'scale_pos_weight (train) = {spw:.3f}')

## 3. Embedding cache + extractor helpers
Each encoder writes one CSV per folder: `{folder}_emb_{name}.csv` with columns `filename, e_0, ..., e_{D-1}`. Existing rows are reused; only missing filenames are extracted.

In [ ]:
def emb_cache_path(folder_name, tag):
    return NB_DIR / f'{folder_name}_emb_{tag}.csv'

def load_cache(folder_name, tag):
    p = emb_cache_path(folder_name, tag)
    if p.exists():
        try:
            return pd.read_csv(p)
        except Exception as e:
            print(f'  [cache] failed to read {p}: {e}')
    return None

def save_cache(folder_name, tag, df):
    p = emb_cache_path(folder_name, tag)
    df.to_csv(p, index=False)
    print(f'  [cache] wrote {p.name}  ({len(df)} rows x {df.shape[1]-1} dims)')

def load_audio_16k(path, sr=16000):
    try:
        y, src_sr = sf.read(str(path), always_2d=False)
        if y.ndim > 1:
            y = y.mean(axis=1)
        y = y.astype(np.float32)
        if src_sr != sr:
            y = librosa.resample(y, orig_sr=src_sr, target_sr=sr)
        return y
    except Exception as e:
        print(f'  [audio] failed {path}: {e}')
        return None

def missing_from_cache(folder_df, cache_df):
    """Return rows of folder_df whose filename isn't in cache_df yet."""
    if cache_df is None or 'filename' not in cache_df.columns:
        return folder_df
    return folder_df[~folder_df['filename'].isin(cache_df['filename'])].copy()

def merge_embeddings(folder_df, cache_df):
    """Inner-join folder_df with cache_df on filename; return (X, y, groups)."""
    m = folder_df.merge(cache_df, on='filename', how='inner')
    e_cols = [c for c in m.columns if c.startswith('e_')]
    X = m[e_cols].values.astype(np.float32)
    y = m['label'].values.astype(int)
    g = m['speaker_id'].values
    return X, y, g, m

In [ ]:
def _to_embedding_row(vec, filename):
    row = {'filename': filename}
    row.update({f'e_{i}': float(v) for i, v in enumerate(vec)})
    return row

def extract_with_wav2vec_family(model_name, tag, folder_groups):
    """Shared extractor for WavLM and Wav2Vec2 (both use AutoFeatureExtractor + AutoModel)."""
    from transformers import AutoFeatureExtractor, AutoModel
    proc  = None
    model = None
    for folder_name, folder_df in folder_groups:
        cache = load_cache(folder_name, tag)
        need  = missing_from_cache(folder_df, cache)
        if len(need) == 0:
            print(f'  [{tag}] {folder_name}: all {len(folder_df)} files cached')
            continue
        if model is None:
            print(f'  [{tag}] loading {model_name} ...')
            proc  = AutoFeatureExtractor.from_pretrained(model_name)
            model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()
        print(f'  [{tag}] {folder_name}: extracting {len(need)} / {len(folder_df)} files')
        rows = []
        for _, r in tqdm(need.iterrows(), total=len(need), desc=f'{tag}:{folder_name}'):
            y = load_audio_16k(r['filepath'])
            if y is None or len(y) < 1600:
                continue
            with torch.no_grad():
                inp = proc(y, sampling_rate=16000, return_tensors='pt').to(DEVICE)
                out = model(**inp).last_hidden_state  # (1, T, D)
                vec = out.mean(dim=1).squeeze(0).cpu().numpy()
            rows.append(_to_embedding_row(vec, r['filename']))
        if rows:
            new_df = pd.DataFrame(rows)
            combined = pd.concat([cache, new_df], ignore_index=True) if cache is not None else new_df
            combined = combined.drop_duplicates(subset='filename', keep='last')
            save_cache(folder_name, tag, combined)
    # free gpu memory after this encoder
    if model is not None:
        del model; torch.cuda.empty_cache() if DEVICE == 'cuda' else None

def extract_with_whisper(model_name, tag, folder_groups):
    """Whisper-medium encoder. Chunks audio to 30s, mean-pools encoder hidden states across chunks."""
    from transformers import WhisperProcessor, WhisperModel
    proc = None
    model = None
    chunk = WHISPER_CHUNK_SEC * 16000
    for folder_name, folder_df in folder_groups:
        cache = load_cache(folder_name, tag)
        need  = missing_from_cache(folder_df, cache)
        if len(need) == 0:
            print(f'  [{tag}] {folder_name}: all {len(folder_df)} files cached')
            continue
        if model is None:
            print(f'  [{tag}] loading {model_name} ...')
            proc  = WhisperProcessor.from_pretrained(model_name)
            model = WhisperModel.from_pretrained(model_name).to(DEVICE).eval()
        print(f'  [{tag}] {folder_name}: extracting {len(need)} / {len(folder_df)} files')
        rows = []
        for _, r in tqdm(need.iterrows(), total=len(need), desc=f'{tag}:{folder_name}'):
            y = load_audio_16k(r['filepath'])
            if y is None or len(y) < 1600:
                continue
            segs = [y[i:i+chunk] for i in range(0, max(len(y),1), chunk)] or [y]
            vecs = []
            with torch.no_grad():
                for seg in segs:
                    if len(seg) < 1600:
                        continue
                    feat = proc(seg, sampling_rate=16000, return_tensors='pt').input_features.to(DEVICE)
                    out  = model.encoder(feat).last_hidden_state  # (1, T, D)
                    vecs.append(out.mean(dim=1).squeeze(0).cpu().numpy())
            if not vecs:
                continue
            vec = np.mean(np.stack(vecs, axis=0), axis=0)
            rows.append(_to_embedding_row(vec, r['filename']))
        if rows:
            new_df = pd.DataFrame(rows)
            combined = pd.concat([cache, new_df], ignore_index=True) if cache is not None else new_df
            combined = combined.drop_duplicates(subset='filename', keep='last')
            save_cache(folder_name, tag, combined)
    if model is not None:
        del model; torch.cuda.empty_cache() if DEVICE == 'cuda' else None

## 4. Train / eval helper (XGBoost on embeddings)

In [ ]:
def train_eval(tag, train_emb, test_emb):
    X_tr, y_tr, g_tr, tr_merged = merge_embeddings(train_df, train_emb)
    X_te, y_te, g_te, te_merged = merge_embeddings(test_df,  test_emb)
    if len(X_tr) == 0 or len(X_te) == 0:
        print(f'  [{tag}] empty train or test after merge -- skip'); return None

    sc = StandardScaler().fit(X_tr)
    X_tr_s = sc.transform(X_tr); X_te_s = sc.transform(X_te)

    D = X_tr_s.shape[1]
    params = dict(XGB_PARAMS, colsample_bytree=0.3 if D > 500 else 0.8,
                  scale_pos_weight=spw)
    clf = xgb.XGBClassifier(**params)
    clf.fit(X_tr_s, y_tr)
    proba = clf.predict_proba(X_te_s)[:, 1]

    # best-F1 threshold sweep
    thr_grid = np.arange(0.20, 0.81, 0.02)
    best = None
    for t in thr_grid:
        pred = (proba >= t).astype(int)
        f1 = f1_score(y_te, pred, zero_division=0)
        if best is None or f1 > best['f1']:
            best = {
                'thr': float(t), 'f1': float(f1),
                'precision': float(precision_score(y_te, pred, zero_division=0)),
                'recall':    float(recall_score(y_te, pred, zero_division=0)),
                'accuracy':  float(accuracy_score(y_te, pred)),
                'pred': pred,
            }
    try:
        auroc = roc_auc_score(y_te, proba)
    except Exception:
        auroc = float('nan')
    tn, fp, fn, tp = confusion_matrix(y_te, best['pred'], labels=[0,1]).ravel()

    # 0.05-step sweep for visibility
    sweep = []
    for t in np.arange(0.20, 0.81, 0.05):
        pred = (proba >= t).astype(int)
        sweep.append({
            'thr': round(float(t),2),
            'prec': round(precision_score(y_te, pred, zero_division=0),4),
            'rec':  round(recall_score(y_te, pred, zero_division=0),4),
            'f1':   round(f1_score(y_te, pred, zero_division=0),4),
        })
    sweep_df = pd.DataFrame(sweep)

    print(f"\n  === {tag}  (D={D}, n_tr={len(X_tr)}, n_te={len(X_te)}) ===")
    print(f"  thr={best['thr']:.2f}  prec={best['precision']:.4f}  rec={best['recall']:.4f}  "
          f"f1={best['f1']:.4f}  acc={best['accuracy']:.4f}  auroc={auroc:.4f}")
    print(f"  tp={tp}  fp={fp}  fn={fn}  tn={tn}")
    print('  threshold sweep (0.05 step):')
    print(sweep_df.to_string(index=False))

    # persist model + predictions
    joblib.dump({'clf': clf, 'scaler': sc, 'dim': D, 'best': {k:v for k,v in best.items() if k!='pred'}},
                SAVE_DIR / f'model_{tag}.pkl')
    pred_df = te_merged[['filename','speaker_id','label']].copy()
    pred_df['proba'] = proba
    pred_df['pred']  = best['pred']
    pred_df.to_csv(SAVE_DIR / f'pred_{tag}.csv', index=False)
    sweep_df.to_csv(SAVE_DIR / f'sweep_{tag}.csv', index=False)

    return {'tag': tag, 'dim': D,
            'n_train': len(X_tr), 'n_test': len(X_te),
            'thr': best['thr'], 'precision': best['precision'],
            'recall': best['recall'], 'f1': best['f1'],
            'accuracy': best['accuracy'], 'auroc': auroc,
            'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn)}

## 5. Run all three encoders sequentially
Per encoder: extract missing → load cached rows for each folder → train → evaluate → free GPU.

In [ ]:
train_folder_groups = [(n, g) for n, g in train_df.groupby('folder')]
test_folder_groups  = [(n, g) for n, g in test_df.groupby('folder')]

results = []
for tag, model_name in MODELS:
    print(f'\n==================================================')
    print(f'  ENCODER: {tag}  ({model_name})')
    print(f'==================================================')
    if tag == 'whisper':
        extract_with_whisper(model_name, tag, train_folder_groups + test_folder_groups)
    else:
        extract_with_wav2vec_family(model_name, tag, train_folder_groups + test_folder_groups)

    train_emb = pd.concat(
        [load_cache(n, tag) for n, _ in train_folder_groups if load_cache(n, tag) is not None],
        ignore_index=True)
    test_emb  = pd.concat(
        [load_cache(n, tag) for n, _ in test_folder_groups  if load_cache(n, tag) is not None],
        ignore_index=True)

    r = train_eval(tag, train_emb, test_emb)
    if r is not None:
        results.append(r)

## 6. Side-by-side comparison

In [ ]:
if results:
    cmp_df = pd.DataFrame(results).sort_values('f1', ascending=False).reset_index(drop=True)
    print(cmp_df.to_string(index=False))
    cmp_df.to_csv(SAVE_DIR / 'encoder_comparison.csv', index=False)
    print(f'\nSaved -> {SAVE_DIR / "encoder_comparison.csv"}')
else:
    print('No results produced.')